# Audio Training

Train a Model with Audio MNIST data and analyse the results

In [2]:
from sdoml_task1.config import PROJECT_DIR, DATA_DIR, N_MFCC
from sdoml_task1.features import extract_features, decode_audio, build_feature_dataset
from sdoml_task1.dataset import AudioMNISTFeaturesDataset
from sdoml_task1.modeling.model import Net
from sdoml_task1.modeling.train import train

from datasets import load_dataset, Audio
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pickle
import torch

In [3]:
ds = load_dataset("gilkeyio/AudioMNIST")
ds = ds.cast_column("audio", Audio(decode=False))

X_train, y_train = build_feature_dataset(ds["train"])
X_test, y_test = build_feature_dataset(ds["test"])

print(X_train.shape, y_train.shape)  # ex: (24000, 26) (24000,)

Extract features: 100%|██████████| 6000/6000 [00:20<00:00, 294.80it/s]

(24000, 26) (24000,)


In [4]:
train_dataset = AudioMNISTFeaturesDataset(X_train, y_train)
test_dataset = AudioMNISTFeaturesDataset(X_test, y_test)

# Data loaders
loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

### MODEL DEFINITION

For training we will use a Multilayer Perceptron with an input of 26 neurons, two hidden layers of 64 and 32 neurons and a final output layer of 10 neurons according to the number of classes in the dataset (0-9 digits). 

More details:
- Optimizer: Adam (lr = 1e-4)
- loss function: Cross Entropy Loss

In [1]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = Net(input_dim=X_train.shape[1]).to(device)

epochs = 40

history_dict = train(model=net, train_loader=loader, val_loader=test_loader, epochs=epochs, lr=1e-4)



NameError: name 'torch' is not defined

In [8]:
display(history_dict)

save_dir = Path(PROJECT_DIR) / "models"
save_dir.mkdir(parents=True, exist_ok=True)

net = history_dict["model"]
torch.save(net.state_dict(), save_dir / "model.pt")

history = {
    "total_loss_train": history_dict["loss_train"],
    "total_accuracy_train": history_dict["accuracy_train"],
    "total_loss_valid": history_dict["loss_val"],
    "total_accuracy_valid": history_dict["accuracy_val"],
    "all_confusion_matrices": history_dict["confusion_matrices"],
    "input_dim": X_train.shape[1],
}

with open(save_dir / "history.pkl", "wb") as f:
    pickle.dump(history, f)

print("model and history saved")

{'loss_train': [11.859962801615398,
  1.9395919389724732,
  1.1812171692848206,
  0.8024646251996358,
  0.5911850216388702,
  0.4691551846663157,
  0.392447372674942,
  0.3413784462213516,
  0.3033703228632609,
  0.2757410843372345,
  0.253771791656812,
  0.23709638273715972,
  0.2223318737546603,
  0.21139054083824158,
  0.2008101093173027,
  0.192133203625679,
  0.18522763055562974,
  0.17999396232763926,
  0.17314115146795908,
  0.168777777582407,
  0.1642227828403314,
  0.16121141595641772,
  0.15663898886243502,
  0.15370440606276195,
  0.15068040430545807,
  0.14646633046865462,
  0.14470645036300023,
  0.14434251606464385,
  0.140317086627086,
  0.13850272076328596,
  0.13564958719412487,
  0.13607570871710778,
  0.1332157927552859,
  0.13090879452228546,
  0.13000693625211715,
  0.1282351550658544,
  0.1270426796277364,
  0.125356750279665,
  0.12459360781311989,
  0.12361440261205038],
 'accuracy_train': [0.138375,
  0.374875,
  0.5832916666666667,
  0.7284166666666667,
  0.81

model and history saved
